# 1. Tải các thư viện liên quan

In [1]:
# Check Torch and CUDA version
import torch
print(f"Torch version {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

Torch version 2.11.0+cu128
CUDA: True


In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn

In [ ]:
!pip uninstall torchvision -y

# 2. Config các thông số của mô hình và thông số HuggingFace

In [4]:
import os
import numpy as np
import evaluate
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

# --- Configuration  ---
MODEL_NAME = "distilbert/distilbert-base-uncased" # Tên model
DATASET_NAME = "stanfordnlp/imdb" # Tên dataset
HF_USERNAME = "TrinhHoangKhang" # Username Hugginface
MODEL_REPO_NAME = "imdb-sentiment-with-distilbert" # Tên repo model trong huggingface
SEED = 42

HUB_MODEL_ID = f"{HF_USERNAME}/{MODEL_REPO_NAME}"
set_seed(SEED)

# Training hyperparameters
NUM_EPOCHS = 4
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
MAX_LENGTH = 256

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
from huggingface_hub import login
login()

# 3. Load dataset - Chia Train/Val/Test

Dataset IMDB bao gồm hai tập train/test với số mẫu dữ liệu 25k/25k

Chúng ta sẽ gộp hai tập này lại và chia train/val/test với tỷ lệ 8/1/1

In [3]:
raw_dataset = load_dataset(DATASET_NAME)

# Combine original train and test splits, then create an 8:1:1 split
full_dataset = concatenate_datasets([raw_dataset["train"], raw_dataset["test"]])
full_dataset = full_dataset.shuffle(seed=SEED)

splits = full_dataset.train_test_split(test_size=0.2, seed=SEED)
train_dataset = splits["train"]
temp_dataset = splits["test"]

val_test = temp_dataset.train_test_split(test_size=0.5, seed=SEED)
val_dataset = val_test["train"]
test_dataset = val_test["test"]

print(f"Train samples: {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")
print(f"Test samples: {len(test_dataset):,}")
print(f"Total samples: {len(full_dataset):,}")
print(f"Split ratio: {len(train_dataset)/len(full_dataset):.1%} / {len(val_dataset)/len(full_dataset):.1%} / {len(test_dataset)/len(full_dataset):.1%}")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Train samples: 40,000
Validation samples: 5,000
Test samples: 5,000
Total samples: 50,000
Split ratio: 80.0% / 10.0% / 10.0%


# 4. Load và sử dụng bộ Tokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

for ds in (tokenized_train, tokenized_val, tokenized_test):
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

# 5. Load mô hình và định nghĩa metric để đánh giá

Ở đây ta sử dụng hai metric là Accuracy và F1

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"],
    }

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# 6. Huấn luyện mô hình

In [7]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    push_to_hub=False,
    hub_model_id=HUB_MODEL_ID,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.236820,0.222012,0.911600,0.911580
2,0.147161,0.313833,0.901400,0.901098
3,0.079489,0.353014,0.917800,0.917791
4,0.047335,0.409627,0.916000,0.915990


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=10000, training_loss=0.14690903208255768, metrics={'train_runtime': 871.5877, 'train_samples_per_second': 183.573, 'train_steps_per_second': 11.473, 'total_flos': 1.0596965513040576e+16, 'train_loss': 0.14690903208255768, 'epoch': 4.0})

# Đánh giá kết quả trên tập test

In [11]:
test_results = trainer.evaluate(tokenized_test)
print("Test set results (Summary):")
for key, value in test_results.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

Test set results (Summary):
  eval_loss: 0.3526
  eval_accuracy: 0.9192
  eval_f1: 0.9192
  eval_runtime: 8.7924
  eval_samples_per_second: 568.6730
  eval_steps_per_second: 17.8560
  epoch: 4.0000


In [12]:
print("\n" + "="*60)

# ----- create classification_report ------
import numpy as np
from sklearn.metrics import classification_report

print("BÁO CÁO PHÂN LOẠI CHI TIẾT (CLASSIFICATION REPORT):")
print("-" * 60)

# get logits
predictions_output = trainer.predict(tokenized_test)

# turn logits into prediction
preds = np.argmax(predictions_output.predictions, axis=-1)
labels = predictions_output.label_ids

# print the report
print(classification_report(labels, preds, target_names=['Negative (0)', 'Positive (1)']))
print("="*60)


BÁO CÁO PHÂN LOẠI CHI TIẾT (CLASSIFICATION REPORT):
------------------------------------------------------------


              precision    recall  f1-score   support

Negative (0)       0.92      0.92      0.92      2514
Positive (1)       0.92      0.92      0.92      2486

    accuracy                           0.92      5000
   macro avg       0.92      0.92      0.92      5000
weighted avg       0.92      0.92      0.92      5000



# 8. Đẩy model lên HuggingFace

In [13]:
trainer.push_to_hub(
    commit_message="Fine-tune DistilBERT on IMDB sentiment classification",
    tags=["text-classification", "sentiment-analysis", "imdb", "distilbert"],
)

print(f"Mô hình được đẩy lên tại: https://huggingface.co/{HUB_MODEL_ID}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...results/model.safetensors:   0%|          |  575kB /  268MB            

  ...results/training_args.bin:   2%|1         |  86.0B / 5.20kB            

Mô hình được đẩy lên tại: https://huggingface.co/TrinhHoangKhang/imdb-sentiment-with-distilbert


# 9. Thử inference trên một số ví dụ

In [14]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=HUB_MODEL_ID,
    tokenizer=HUB_MODEL_ID,
    device=0 if device == "cuda" else -1,
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

In [15]:
def predict_sentiment(review):
    # Lấy kết quả đầu tiên từ pipeline
    prediction = classifier(review)[0]
    raw_label = prediction['label']
    score = prediction['score']

    # Ánh xạ từ LABEL_1/LABEL_0 sang chữ tường minh
    if raw_label == "LABEL_1":
        human_label = "Positive"
    else:
        human_label = "Negative"

    print("Review: ", review)
    print(f">> Sentiment: {human_label} | Độ tự tin: {score:.2%}")
    print("\n")

# Nhóm tích cực
positive_reviews = [
    "This movie was absolutely fantastic! The acting and storyline were brilliant.",
    "I went in with low expectations but was blown away. Truly one of the best films of the year."
]
# Nhóm tiêu cực
negative_reviews = [
    "Total disaster. The plot makes no sense, the acting is wooden, and I fell asleep halfway through.",
    "Don't waste your time or money on this rubbish. A complete disappointment from start to finish."
]
# Nhóm mơ hồ
ambiguous_reviews = [
    "The acting was okay, the music was nice, but the story was just average. Nothing special, but not bad either.",
    "The first half of the film was a absolute 10/10 masterpiece, but the second half completely dragged it down to a 2/10 disaster."
]

print("==== Nhóm tích cực: ====")
for review in positive_reviews:
    predict_sentiment(review)

print('==== Nhóm tiêu cực: ===')
for review in negative_reviews:
    predict_sentiment(review)

print('==== Nhóm mơ hồ: ===')
for review in ambiguous_reviews:
    predict_sentiment(review)

==== Nhóm tích cực: ====
Review:  This movie was absolutely fantastic! The acting and storyline were brilliant.
>> Sentiment: Positive | Độ tự tin: 99.78%


Review:  I went in with low expectations but was blown away. Truly one of the best films of the year.
>> Sentiment: Positive | Độ tự tin: 99.77%


==== Nhóm tiêu cực: ===
Review:  Total disaster. The plot makes no sense, the acting is wooden, and I fell asleep halfway through.
>> Sentiment: Negative | Độ tự tin: 99.84%


Review:  Don't waste your time or money on this rubbish. A complete disappointment from start to finish.
>> Sentiment: Negative | Độ tự tin: 99.87%


==== Nhóm mơ hồ: ===
Review:  The acting was okay, the music was nice, but the story was just average. Nothing special, but not bad either.
>> Sentiment: Negative | Độ tự tin: 99.03%


Review:  The first half of the film was a absolute 10/10 masterpiece, but the second half completely dragged it down to a 2/10 disaster.
>> Sentiment: Negative | Độ tự tin: 99.44%


